In [17]:
pip install notebook

Note: you may need to restart the kernel to use updated packages.


In [18]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [19]:
from pyspark.sql import SparkSession

In [20]:
spark = SparkSession.builder \
    .appName("PySpark-jupyter-demo") \
    .getOrCreate()


In [21]:
data = [("Kia", "Niro", 2025), ("Toyota", "Rav4", 2019), ("BYD", "Atto3", 2024)]
df = spark.createDataFrame(data, ["Car", "Model", "Year"])
df.show()

+------+-----+----+
|   Car|Model|Year|
+------+-----+----+
|   Kia| Niro|2025|
|Toyota| Rav4|2019|
|   BYD|Atto3|2024|
+------+-----+----+



In [22]:
df_lines = spark.read.format("com.databricks.spark.csv").options(header='true') \
.option("delimiter", ",").option("quote", '"') \
.option("escape", '"').option("multiLine", True) \
.load("simpsons/simpsons_script_lines.csv")

In [23]:
# create lines table and the additional tables needed by later queries
df_lines.createOrReplaceTempView("lines_table")

# load the other CSVs and create temp views so locations_table, episodes_table, characters_table exist
# use relative paths (will work on Linux Debian when run from the repo root)
df_locations = spark.read.csv("simpsons/simpsons_locations.csv", header=True, inferSchema=True)
df_locations.createOrReplaceTempView("locations_table")

df_episodes = spark.read.csv("simpsons/simpsons_episodes.csv", header=True, inferSchema=True)
df_episodes.createOrReplaceTempView("episodes_table")

df_characters = spark.read.csv("simpsons/simpsons_characters.csv", header=True, inferSchema=True)
df_characters.createOrReplaceTempView("characters_table")

In [24]:
df_lines.createOrReplaceTempView("lines_table")

In [25]:
## print lines file schema
df_lines.printSchema()

root
 |-- id: string (nullable = true)
 |-- episode_id: string (nullable = true)
 |-- number: string (nullable = true)
 |-- raw_text: string (nullable = true)
 |-- timestamp_in_ms: string (nullable = true)
 |-- speaking_line: string (nullable = true)
 |-- character_id: string (nullable = true)
 |-- location_id: string (nullable = true)
 |-- raw_character_text: string (nullable = true)
 |-- raw_location_text: string (nullable = true)
 |-- spoken_words: string (nullable = true)
 |-- normalized_text: string (nullable = true)
 |-- word_count: string (nullable = true)



In [26]:
## print all fields for first 5 rows
spark.sql("SELECT * from lines_table").show(5)

+----+----------+------+--------------------+---------------+-------------+------------+-----------+--------------------+--------------------+--------------------+--------------------+----------+
|  id|episode_id|number|            raw_text|timestamp_in_ms|speaking_line|character_id|location_id|  raw_character_text|   raw_location_text|        spoken_words|     normalized_text|word_count|
+----+----------+------+--------------------+---------------+-------------+------------+-----------+--------------------+--------------------+--------------------+--------------------+----------+
|9549|        32|   209|Miss Hoover: No, ...|         848000|         true|         464|          3|         Miss Hoover|Springfield Eleme...|No, actually, it ...|no actually it wa...|        31|
|9550|        32|   210|Lisa Simpson: (NE...|         856000|         true|           9|          3|        Lisa Simpson|Springfield Eleme...|Where's Mr. Bergs...| wheres mr bergstrom|         3|
|9551|        32|   

In [27]:
## print the shortest 10 script lines:
spark.sql("SELECT int(word_count) AS wc, raw_text FROM lines_table WHERE TRY_CAST(word_count AS INT) > 0 ORDER BY wc ").show(10,False)

[Stage 18:>                                                         (0 + 1) / 1]

+---+---------------------------------+
|wc |raw_text                         |
+---+---------------------------------+
|1  |Homer Simpson: Oh.               |
|1  |Homer Simpson: And?              |
|1  |Bart Simpson: Lewis?             |
|1  |Homer Simpson: (SHOCKED) Me?     |
|1  |Wendell Borton: Yayyyyyyyyyyyyyy!|
|1  |Lisa Simpson: Baboon!            |
|1  |Lisa Simpson: Yeah.              |
|1  |Lisa Simpson: No!                |
|1  |Homer Simpson: Oh.               |
|1  |Homer Simpson: Nuts.             |
+---+---------------------------------+
only showing top 10 rows


In [28]:
## count the scenes that took more than 10 minutes
spark.sql("SELECT count(*) FROM lines_table WHERE TRY_CAST(timestamp_in_ms AS INT) > (10 * 60 * 1000)").show()

[Stage 19:>                                                         (0 + 1) / 1]

+--------+
|count(1)|
+--------+
|   86678|
+--------+



In [29]:
## print the first 50 locations that has the word "Springfield" in them , ignoring letters case.
spark.sql("SELECT * FROM locations_table WHERE LOWER(name) LIKE '%springfield%' LIMIT 50").show(50,False)

[Stage 22:>                                                         (0 + 1) / 1]

+---+-----------------------------------------------+-----------------------------------------------+
|id |name                                           |normalized_name                                |
+---+-----------------------------------------------+-----------------------------------------------+
|3  |Springfield Elementary School                  |springfield elementary school                  |
|8  |Springfield Mall                               |springfield mall                               |
|10 |Springfield Nuclear Power Plant                |springfield nuclear power plant                |
|20 |Springfield Downs Dog Track                    |springfield downs dog track                    |
|21 |SPRINGFIELD DOWNS                              |springfield downs                              |
|23 |SPRINGFIELD DOWN                               |springfield down                               |
|24 |SPRINGFIELD DOWNS PARKING LOT                  |springfield downs parking lot

In [30]:
## print 20 quotes that are located in any place that has Jerusalem in its name.
## Note that Jerusalem may appear in any case and in any part of the location name.
## Use JOIN for this query on another table
spark.sql("SELECT l.raw_text AS quote, loc.name AS location FROM lines_table l JOIN locations_table loc ON TRY_CAST(l.location_id AS INT) = loc.id WHERE LOWER(loc.name) LIKE '%jerusalem%' LIMIT 20").show(20,False)

[Stage 24:>                                                         (0 + 1) / 1]

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------+
|quote                                                                                                                                                                                     |location              |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------+
|(JERUSALEM ROYAL PALACE: ext. jerusalem royal palace - plaza - day)                                                                                                                       |JERUSALEM ROYAL PALACE|
|Bart Simpson: I'm bored. Send in my jester.                                                                                                            

In [31]:
## print first 20 most used locations with the count of lines spoken in them.
## use GROUP BY for that
spark.sql("SELECT loc.name AS location, COUNT(*) AS lines_count FROM lines_table l JOIN locations_table loc ON TRY_CAST(l.location_id AS INT) = loc.id GROUP BY loc.name ORDER BY lines_count DESC LIMIT 20").show(20,False)

[Stage 26:>                                                         (0 + 1) / 1]

+-------------------------------+-----------+
|location                       |lines_count|
+-------------------------------+-----------+
|Simpson Home                   |35059      |
|Springfield Elementary School  |7092       |
|Moe's Tavern                   |4628       |
|Springfield Nuclear Power Plant|3594       |
|Kwik-E-Mart                    |1476       |
|First Church of Springfield    |1416       |
|Simpson Living Room            |1378       |
|Springfield Street             |1334       |
|Springfield                    |1314       |
|Simpson Car                    |1239       |
|Flanders Home                  |1166       |
|Street                         |1124       |
|Springfield Town Hall          |1103       |
|Springfield Retirement Castle  |1049       |
|Burns Manor                    |998        |
|Springfield Mall               |833        |
|Simpson Kitchen                |816        |
|Courtroom                      |813        |
|Bart's Treehouse               |7

In [32]:
## find the seasons in which the average imdb rating was the highest.
## Print the seasons number, the number of episodes in each one and the average rating
## in a descending order from highest average rating to lowest.
spark.sql("SELECT season, COUNT(*) AS episode_count, AVG(TRY_CAST(imdb_rating AS DOUBLE)) AS avg_rating FROM episodes_table WHERE TRY_CAST(imdb_rating AS DOUBLE) IS NOT NULL GROUP BY season ORDER BY avg_rating DESC").show(100,False)

+------+-------------+------------------+
|season|episode_count|avg_rating        |
+------+-------------+------------------+
|5     |22           |8.336363636363636 |
|7     |25           |8.324             |
|6     |25           |8.312             |
|4     |22           |8.268181818181818 |
|8     |25           |8.219999999999999 |
|3     |24           |8.154166666666665 |
|2     |22           |8.04090909090909  |
|9     |25           |7.8439999999999985|
|1     |13           |7.807692307692307 |
|10    |23           |7.569565217391306 |
|12    |21           |7.361904761904761 |
|11    |22           |7.290909090909093 |
|13    |22           |7.140909090909091 |
|14    |22           |7.077272727272727 |
|15    |22           |7.045454545454546 |
|16    |21           |7.042857142857143 |
|18    |22           |7.0               |
|19    |20           |6.935             |
|20    |21           |6.895238095238096 |
|17    |22           |6.863636363636362 |
|25    |22           |6.8318181818